# E2E Simulation: 1000 New Tax Applications

Simulates 1000 new tax applications end-to-end:

| Step | Description |
|------|-------------|
| 1 | Generate 1000 synthetic filings (70% clean, 20% shared-address, 10% shared-IP) |
| 2 | Ingest via Stream Feature View REST API (batches of 50) |
| 3 | Score via Gateway (`FRAUD_GATEWAY`) with 10 concurrent workers |
| 4 | Report decisions, detection rates, latency, and cost |

**Prerequisites:** Notebooks 00–03 completed (Feature Store, model, Gateway deployed).

In [ ]:
import os, time, random, json
import numpy as np
import pandas as pd
import requests
import tomllib, pathlib
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import snowflake.connector
from snowflake.snowpark import Session

# Load PAT from Snowflake config
_sf_config = tomllib.load(open(pathlib.Path.home() / '.snowflake' / 'config.toml', 'rb'))
os.environ.setdefault('SNOWFLAKE_PAT', _sf_config['connections']['parker_demo']['password'])
PAT = os.environ['SNOWFLAKE_PAT']
print(f'SNOWFLAKE_PAT loaded ({len(PAT)} chars)')

# Session
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    conn = snowflake.connector.connect(connection_name='parker_demo')
    session = Session.builder.configs({'connection': conn}).create()

session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql('USE DATABASE GRAPH_FRAUD_DEMO').collect()
session.sql('USE WAREHOUSE GRAPH_FRAUD_WH').collect()
print(f'Connected: {session.get_current_account()}')

In [ ]:
# Config
DATABASE       = 'GRAPH_FRAUD_DEMO'
FEATURE_SCHEMA = 'FEATURES'
MODEL_SCHEMA   = 'MODELS'
WAREHOUSE      = 'GRAPH_FRAUD_WH'
GATEWAY_NAME   = 'GRAPH_FRAUD_GATEWAY'
STREAM_SOURCE  = 'APPLICATION_EVENTS'

MODEL_FEATURES = [
    'EFFECTIVE_COMMUNITY_SIZE', 'DEGREE', 'SHARED_ADDRESS_COUNT', 'SHARED_IP_COUNT',
    'AVG_COMMUNITY_REFUND', 'COMMUNITY_REFUND_STDDEV',
    'ADDR_N_FILERS', 'ADDR_AVG_REFUND', 'IP_N_FILERS', 'IP_AVG_REFUND',
    'SUBMISSION_HOUR', 'IS_HIGH_CLAIM', 'IS_ODD_HOUR', 'CLAIM_AMOUNT', 'HAS_DIRECT_GRAPH',
]

HEADERS = {
    'Authorization': f'Snowflake Token="{PAT}"',
    'Content-Type': 'application/json',
}

# Resolve OFS endpoints (import online_service here — top-level import hangs)
from snowflake.ml.feature_store import FeatureStore, CreationMode, online_service

fs = FeatureStore(
    session=session, database=DATABASE, name=FEATURE_SCHEMA,
    default_warehouse=WAREHOUSE, creation_mode=CreationMode.FAIL_IF_NOT_EXIST,
)

status = fs.get_online_service_status()
INGEST_URL = online_service.endpoint_url(status, 'ingest')
QUERY_URL  = online_service.endpoint_url(status, 'query')
print(f'OFS status : {status.status}')
print(f'Ingest URL : {INGEST_URL}')
print(f'Query URL  : {QUERY_URL}')

# Resolve Gateway URL
rows = session.sql(f'DESC GATEWAY {DATABASE}.{MODEL_SCHEMA}.{GATEWAY_NAME}').collect()
GATEWAY_URL = ''
for row in rows:
    d = row.as_dict()
    url = d.get('ingress_url', '').strip()
    if url and '.' in url:
        GATEWAY_URL = f'https://{url}' if not url.startswith('http') else url
        break
print(f'Gateway URL: {GATEWAY_URL}')


In [ ]:
# Column orders per FV (OFS REST returns positional arrays, no column names)
FV_COLUMNS = {
    'GRAPH_COMMUNITY_FEATURES': [
        'COMMUNITY_ID', 'COMMUNITY_SIZE', 'IS_LARGE_COMMUNITY', 'DEGREE',
        'SHARED_ADDRESS_COUNT', 'SHARED_IP_COUNT', 'SHARED_DEVICE_COUNT', 'SHARED_BANK_COUNT',
        'AVG_COMMUNITY_REFUND', 'COMMUNITY_REFUND_STDDEV',
        'ADDR_N_FILERS', 'ADDR_COMMUNITY_SIZE', 'ADDR_AVG_REFUND',
        'IP_N_FILERS', 'IP_COMMUNITY_SIZE', 'IP_AVG_REFUND',
    ],
    'ADDRESS_COMMUNITY_FEATURES': [
        'NUM_FILERS_AT_ADDRESS', 'COMMUNITY_SIZE_AT_ADDRESS', 'AVG_REFUND_AT_ADDRESS',
    ],
    'IP_COMMUNITY_FEATURES': [
        'NUM_FILERS_AT_IP', 'COMMUNITY_SIZE_AT_IP', 'AVG_REFUND_AT_IP',
    ],
}

def query_ofs(fv_name, fv_version, entity_key_name, entity_key_value):
    """Read one entity from OFS REST Query API. Returns (features_dict, latency_ms)."""
    t0 = time.time()
    resp = requests.post(
        f'{QUERY_URL}/api/v1/query',
        headers=HEADERS,
        json={
            'name': fv_name, 'version': fv_version,
            'object_type': 'feature_view',
            'request_rows': [{'entity': {entity_key_name: str(entity_key_value)}}],
        },
        timeout=5,
    )
    ms = (time.time() - t0) * 1000
    row = {}
    if resp.status_code == 200:
        data = resp.json()
        results = data.get('results', data.get('rows', []))
        if results:
            feats = results[0].get('features', {})
            if isinstance(feats, dict):
                row = feats
            elif isinstance(feats, list):
                cols = FV_COLUMNS.get(fv_name, [])
                if len(cols) == len(feats):
                    row = dict(zip(cols, feats))
                else:
                    row = {f'f{i}': v for i, v in enumerate(feats)}
    return row, ms


def ingest_one(app):
    """Fire-and-forget ingest a single filing event for next-request velocity."""
    event = {
        'APPLICANT_ID':       str(app['APPLICANT_ID']),
        'ADDRESS_ID': str(app['ADDRESS_ID']),
        'IP_ADDRESS':        str(app['IP_ADDRESS']),
        'CLAIM_AMOUNT':     float(app['CLAIM_AMOUNT']),
        'SUBMISSION_HOUR':       float(app['SUBMISSION_HOUR']),
        'IS_HIGH_CLAIM':    float(app['IS_HIGH_CLAIM']),
        'IS_ODD_HOUR':       float(app['IS_ODD_HOUR']),
        'SUBMITTED_AT':         app['SUBMITTED_AT'],
    }
    try:
        resp = requests.post(
            f'{INGEST_URL}/api/v1/ingest',
            headers=HEADERS,
            json={'records': {STREAM_SOURCE: [event]}},
            timeout=10,
        )
        if resp.status_code != 200 and len(ingest_errors) < 3:
            ingest_errors.append(f'{resp.status_code}: {resp.text[:200]}')
    except Exception as e:
        if len(ingest_errors) < 3:
            ingest_errors.append(str(e)[:200])

# Quick test: ingest one event to verify it works
test_event = {
    'APPLICANT_ID': '9999999',
    'ADDRESS_ID': '1',
    'IP_ADDRESS': '1.2.3.4',
    'CLAIM_AMOUNT': 1000.0,
    'SUBMISSION_HOUR': 10.0,
    'IS_HIGH_CLAIM': 0.0,
    'IS_ODD_HOUR': 0.0,
    'SUBMITTED_AT': '2026-10-14T12:00:00',
}
resp = requests.post(
    f'{INGEST_URL}/api/v1/ingest',
    headers=HEADERS,
    json={'records': {STREAM_SOURCE: [test_event]}},
    timeout=10,
)
print(f'Ingest test: {resp.status_code}')
if resp.status_code != 200:
    print(f'  Error: {resp.text[:300]}')
else:
    print(f'  OK: {resp.json()}')

# Quick test: query_ofs
test_feats, test_ms = query_ofs('ADDRESS_COMMUNITY_FEATURES', 'V1', 'ADDRESS_ID', '1')
print(f'OFS query test: {test_ms:.0f}ms, features: {test_feats}')



In [ ]:
# Generate 1000 synthetic tax applications
# 70% clean, 20% shared-address with fraud ring, 10% shared-IP with fraud ring

# Fetch fraud ring shared identifiers for suspicious applications
ring_rows = session.sql(f'''
    SELECT RING_ID, RING_ADDRESS_ID, RING_IP
    FROM {DATABASE}.RAW.FRAUD_RINGS
    WHERE RING_ADDRESS_ID IS NOT NULL OR RING_IP IS NOT NULL
''').collect()

ring_addresses = [str(r['RING_ADDRESS_ID']) for r in ring_rows if r['RING_ADDRESS_ID']]
ring_ips       = [str(r['RING_IP']) for r in ring_rows if r['RING_IP']]
print(f'Fraud rings: {len(ring_addresses)} shared addresses, {len(ring_ips)} shared IPs')

np.random.seed(42)
random.seed(42)

N_APPS = 100
applications = []
now_ts = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')

for i in range(N_APPS):
    tid = 9100001 + i
    r = random.random()

    if r < 0.70:
        # Clean application
        app = {
            'APPLICANT_ID': str(tid),
            'ADDRESS_ID': f'ADDR_{random.randint(100000, 999999)}',
            'IP_ADDRESS': f'{random.randint(1,223)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}',
            'CLAIM_AMOUNT': float(np.random.uniform(1000, 5000)),
            'SUBMISSION_HOUR': float(np.random.randint(8, 18)),
            'TYPE': 'clean',
        }
    elif r < 0.90:
        # Shared address with a fraud ring
        app = {
            'APPLICANT_ID': str(tid),
            'ADDRESS_ID': random.choice(ring_addresses) if ring_addresses else f'ADDR_{random.randint(1,100)}',
            'IP_ADDRESS': f'{random.randint(1,223)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}',
            'CLAIM_AMOUNT': float(np.random.uniform(8000, 25000)),
            'SUBMISSION_HOUR': float(np.random.choice([1, 2, 3, 4, 23])),
            'TYPE': 'suspicious_addr',
        }
    else:
        # Shared IP with a fraud ring
        app = {
            'APPLICANT_ID': str(tid),
            'ADDRESS_ID': f'ADDR_{random.randint(100000, 999999)}',
            'IP_ADDRESS': random.choice(ring_ips) if ring_ips else f'10.0.0.{random.randint(1,10)}',
            'CLAIM_AMOUNT': float(np.random.uniform(8000, 25000)),
            'SUBMISSION_HOUR': float(np.random.choice([1, 2, 3, 4, 23])),
            'TYPE': 'suspicious_ip',
        }

    app['SUBMITTED_AT'] = now_ts
    app['IS_HIGH_CLAIM'] = 1.0 if app['CLAIM_AMOUNT'] > 7000 else 0.0
    app['IS_ODD_HOUR'] = 1.0 if app['SUBMISSION_HOUR'] < 6 or app['SUBMISSION_HOUR'] > 22 else 0.0
    applications.append(app)

types = pd.Series([a['TYPE'] for a in applications]).value_counts()
print(f'\nGenerated {N_APPS} applications:')
for t, n in types.items():
    print(f'  {t}: {n}')

In [ ]:
# Quick check: do the OFS feature views return data for our synthetic applicants?
tid = applications[0]['APPLICANT_ID']
aid = applications[0]['ADDRESS_ID']
ip = applications[0]['IP_ADDRESS']
print(f'Testing applicant: {tid}, address: {aid}, IP: {ip}')

# Graph FV (applicant-keyed, auto-fetched by Gateway)
graph_feats, graph_ms = query_ofs('GRAPH_COMMUNITY_FEATURES', 'V1', 'APPLICANT_ID', str(tid))
print(f'\nGraph FV ({graph_ms:.0f}ms): {graph_feats}')

# Address FV
addr_feats, addr_ms = query_ofs('ADDRESS_COMMUNITY_FEATURES', 'V1', 'ADDRESS_ID', str(aid))
print(f'Address FV ({addr_ms:.0f}ms): {addr_feats}')

# IP FV
ip_feats, ip_ms = query_ofs('IP_COMMUNITY_FEATURES', 'V1', 'IP_ADDRESS', str(ip))
print(f'IP FV ({ip_ms:.0f}ms): {ip_feats}')


In [ ]:
# Snapshot pre-simulation credits
pre_credits = session.sql('''
    SELECT SUM(CREDITS_USED) AS TOTAL_CREDITS
    FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_HISTORY
    WHERE START_TIME >= DATEADD('hour', -2, CURRENT_TIMESTAMP())
''').collect()
credits_before = float(pre_credits[0]['TOTAL_CREDITS'] or 0)
sim_start_ts = datetime.now(timezone.utc)
print(f'Credits used (last 2h before sim): {credits_before:.4f}')
print(f'Simulation start: {sim_start_ts.strftime("%H:%M:%S UTC")}')

In [ ]:
import time
import requests
import pandas as pd
from datetime import datetime, timezone

# === 1. Generate one new suspicious application ===
new_app = {
    'APPLICANT_ID': '9999999',
    'CLAIM_AMOUNT': 18500.00,
}

# Grab a fraud ring address/IP so the heuristic features fire
ring = session.sql("SELECT RING_ADDRESS_ID, RING_IP FROM GRAPH_FRAUD_DEMO.RAW.FRAUD_RINGS WHERE RING_ADDRESS_ID IS NOT NULL LIMIT 1").collect()[0]
new_app['ADDRESS_ID'] = str(ring['RING_ADDRESS_ID'])
new_app['IP_ADDRESS'] = str(ring['RING_IP'])

print(f"=== New Application ===")
print(f"  Applicant: {new_app['APPLICANT_ID']}")
print(f"  Claim:   ${new_app['CLAIM_AMOUNT']:,.2f}")
print(f"  Address:  {new_app['ADDRESS_ID']} (fraud ring address)")
print(f"  IP:       {new_app['IP_ADDRESS']} (fraud ring IP)")

# === 2. Ingest into stream FV (async, for velocity features on next request) ===
ingest_event = {
    'APPLICANT_ID': str(new_app['APPLICANT_ID']),
    'ADDRESS_ID': str(new_app['ADDRESS_ID']),
    'IP_ADDRESS': new_app['IP_ADDRESS'],
    'CLAIM_AMOUNT': float(new_app['CLAIM_AMOUNT']),
    'SUBMISSION_HOUR': float(datetime.now().hour),
    'IS_HIGH_CLAIM': 1.0 if new_app['CLAIM_AMOUNT'] > 10000 else 0.0,
    'IS_ODD_HOUR': 1.0 if datetime.now().hour <= 5 else 0.0,
    'SUBMITTED_AT': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S'),
}

t0 = time.time()
ingest_resp = requests.post(
    f'{INGEST_URL}/api/v1/ingest',
    headers=HEADERS,
    json={'records': {STREAM_SOURCE: [ingest_event]}},
    timeout=10,
)
ingest_ms = (time.time() - t0) * 1000
print(f"\n=== Stream Ingest ===")
print(f"  Status: {ingest_resp.status_code} ({ingest_ms:.0f}ms)")
if ingest_resp.status_code == 200:
    print(f"  Response: {ingest_resp.json()}")
else:
    print(f"  Error: {ingest_resp.text[:200]}")

# === 3. Parallel OFS reads for all 3 feature views ===
from concurrent.futures import ThreadPoolExecutor

t0 = time.time()
with ThreadPoolExecutor(max_workers=3) as ofs_pool:
    graph_fut = ofs_pool.submit(query_ofs, 'GRAPH_COMMUNITY_FEATURES', 'V1', 'APPLICANT_ID', str(new_app['APPLICANT_ID']))
    addr_fut = ofs_pool.submit(query_ofs, 'ADDRESS_COMMUNITY_FEATURES', 'V1', 'ADDRESS_ID', str(new_app['ADDRESS_ID']))
    ip_fut = ofs_pool.submit(query_ofs, 'IP_COMMUNITY_FEATURES', 'V1', 'IP_ADDRESS', str(new_app['IP_ADDRESS']))
    g, g_ms = graph_fut.result()
    a, a_ms = addr_fut.result()
    ip_feats, ip_ms = ip_fut.result()
ofs_ms = (time.time() - t0) * 1000

print(f"\n=== OFS Feature Reads ({ofs_ms:.0f}ms parallel) ===")
print(f"  Graph FV   ({g_ms:.0f}ms): {g}")
print(f"  Address FV ({a_ms:.0f}ms): {a}")
print(f"  IP FV      ({ip_ms:.0f}ms): {ip_feats}")

def n(val, default=0.0):
    try: return float(val) if val is not None else default
    except: return default

community_size = n(g.get('COMMUNITY_SIZE'))
addr_cs = n(a.get('COMMUNITY_SIZE_AT_ADDRESS'))
ip_cs = n(ip_feats.get('COMMUNITY_SIZE_AT_IP'))
submission_hour = float(datetime.now().hour)

# === 4. Score via Gateway — send ALL 15 features (no auto-fetch) ===
features = {
    'EFFECTIVE_COMMUNITY_SIZE': community_size if community_size > 0 else (addr_cs if addr_cs > 0 else ip_cs),
    'DEGREE': n(g.get('DEGREE')),
    'SHARED_ADDRESS_COUNT': n(g.get('SHARED_ADDRESS_COUNT')),
    'SHARED_IP_COUNT': n(g.get('SHARED_IP_COUNT')),
    'AVG_COMMUNITY_REFUND': n(g.get('AVG_COMMUNITY_REFUND')),
    'COMMUNITY_REFUND_STDDEV': n(g.get('COMMUNITY_REFUND_STDDEV')),
    'ADDR_N_FILERS': n(a.get('NUM_FILERS_AT_ADDRESS')),
    'ADDR_AVG_REFUND': n(a.get('AVG_REFUND_AT_ADDRESS')),
    'IP_N_FILERS': n(ip_feats.get('NUM_FILERS_AT_IP')),
    'IP_AVG_REFUND': n(ip_feats.get('AVG_REFUND_AT_IP')),
    'SUBMISSION_HOUR': submission_hour,
    'IS_HIGH_CLAIM': 1.0 if new_app['CLAIM_AMOUNT'] > 10000 else 0.0,
    'IS_ODD_HOUR': 1.0 if submission_hour <= 5 else 0.0,
    'CLAIM_AMOUNT': float(new_app['CLAIM_AMOUNT']),
    'HAS_DIRECT_GRAPH': 1.0 if community_size > 0 else 0.0,
}

t1 = time.time()
score_resp = requests.post(
    f'{GATEWAY_URL}/predict',
    headers=HEADERS,
    json={'dataframe_records': [features]},
    timeout=15,
)
score_ms = (time.time() - t1) * 1000
total_ms = ingest_ms + ofs_ms + score_ms

print(f"\n=== Gateway Scoring ===")
print(f"  Status: {score_resp.status_code} ({score_ms:.0f}ms)")
print(f"  Response: {score_resp.text[:500]}")
print(f"\n=== Total Pipeline: {total_ms:.0f}ms (ingest {ingest_ms:.0f} + OFS {ofs_ms:.0f} + score {score_ms:.0f}) ===")


In [ ]:
import time, requests

# Warm up
for _ in range(3):
    requests.post(f'{QUERY_URL}/api/v1/query', headers=HEADERS,
        json={'name': 'ADDRESS_COMMUNITY_FEATURES', 'version': 'V1',
                'object_type': 'feature_view',
                'request_rows': [{'entity': {'ADDRESS_ID': str(ring['RING_ADDRESS_ID'])}}]},
        timeout=5)

# Benchmark 10 reads
lats = []
for _ in range(10):
    t0 = time.time()
    requests.post(f'{QUERY_URL}/api/v1/query', headers=HEADERS,
        json={'name': 'ADDRESS_COMMUNITY_FEATURES', 'version': 'V1',
                'object_type': 'feature_view',
                'request_rows': [{'entity': {'ADDRESS_ID': str(ring['RING_ADDRESS_ID'])}}]},
        timeout=5)
    lats.append((time.time() - t0) * 1000)

lats.sort()
print(f'OFS REST Query latency (laptop → OFS):')
print(f'  p50: {lats[5]:.0f}ms | p90: {lats[9]:.0f}ms | mean: {sum(lats)/len(lats):.0f}ms')


In [ ]:
# Score each application: parallel OFS reads + Gateway with all 15 features
from concurrent.futures import ThreadPoolExecutor, as_completed

N_WORKERS = 1
async_pool = ThreadPoolExecutor(max_workers=4)
results = []
score_errors = 0

def score_one(app):
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=3) as ofs_pool:
        graph_fut = ofs_pool.submit(query_ofs, 'GRAPH_COMMUNITY_FEATURES', 'V1', 'APPLICANT_ID', str(app['APPLICANT_ID']))
        addr_fut = ofs_pool.submit(query_ofs, 'ADDRESS_COMMUNITY_FEATURES', 'V1', 'ADDRESS_ID', str(app['ADDRESS_ID']))
        ip_fut = ofs_pool.submit(query_ofs, 'IP_COMMUNITY_FEATURES', 'V1', 'IP_ADDRESS', str(app['IP_ADDRESS']))
        g, g_ms = graph_fut.result()
        a, a_ms = addr_fut.result()
        ip_feats, ip_ms = ip_fut.result()

    def n(val, default=0.0):
        try: return float(val) if val is not None else default
        except: return default

    community_size = n(g.get('COMMUNITY_SIZE'))
    addr_cs = n(a.get('COMMUNITY_SIZE_AT_ADDRESS'))
    ip_cs = n(ip_feats.get('COMMUNITY_SIZE_AT_IP'))

    features = {
        'EFFECTIVE_COMMUNITY_SIZE': community_size if community_size > 0 else (addr_cs if addr_cs > 0 else ip_cs),
        'DEGREE': n(g.get('DEGREE')),
        'SHARED_ADDRESS_COUNT': n(g.get('SHARED_ADDRESS_COUNT')),
        'SHARED_IP_COUNT': n(g.get('SHARED_IP_COUNT')),
        'AVG_COMMUNITY_REFUND': n(g.get('AVG_COMMUNITY_REFUND')),
        'COMMUNITY_REFUND_STDDEV': n(g.get('COMMUNITY_REFUND_STDDEV')),
        'ADDR_N_FILERS': n(a.get('NUM_FILERS_AT_ADDRESS')),
        'ADDR_AVG_REFUND': n(a.get('AVG_REFUND_AT_ADDRESS')),
        'IP_N_FILERS': n(ip_feats.get('NUM_FILERS_AT_IP')),
        'IP_AVG_REFUND': n(ip_feats.get('AVG_REFUND_AT_IP')),
        'SUBMISSION_HOUR': float(app.get('SUBMISSION_HOUR', 12)),
        'IS_HIGH_CLAIM': 1.0 if app['CLAIM_AMOUNT'] > 10000 else 0.0,
        'IS_ODD_HOUR': 1.0 if app.get('SUBMISSION_HOUR', 12) <= 5 else 0.0,
        'CLAIM_AMOUNT': float(app['CLAIM_AMOUNT']),
        'HAS_DIRECT_GRAPH': 1.0 if community_size > 0 else 0.0,
    }

    resp = requests.post(f'{GATEWAY_URL}/predict', headers=HEADERS,
        json={'dataframe_records': [features]}, timeout=10)
    lat = (time.time() - t0) * 1000

    prob = None
    if resp.status_code == 200:
        data = resp.json()
        rows = data.get('data', data.get('dataframe_records', []))
        if rows:
            if isinstance(rows[0], dict):
                prob = float(rows[0].get('output_feature_0', 0))
            elif isinstance(rows[0], list) and len(rows[0]) > 1:
                raw = rows[0][1]
                prob = float(raw.get('output_feature_0', 0)) if isinstance(raw, dict) else float(raw)
            else:
                prob = 0.0
        else:
            prob = 0.0

    async_pool.submit(ingest_one, app)
    return app, prob, lat

print(f'Scoring {N_APPS} applications ({N_WORKERS} concurrent workers)...')
print('All 3 FVs read client-side via OFS REST; all 15 features sent to Gateway')
score_t0 = time.time()
score_latencies = []

with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    futures = {pool.submit(score_one, app): app for app in applications}
    done = 0
    for fut in as_completed(futures):
        app, prob, lat = fut.result()
        score_latencies.append(lat)
        if prob is not None:
            decision = 'DECLINE' if prob >= 0.5 else 'APPROVE'
            results.append({
                'APPLICANT_ID': app['APPLICANT_ID'],
                'TYPE': app['TYPE'],
                'PREDICTION': int(prob),
                'DECISION': decision,
            })
        else:
            score_errors += 1
        done += 1
        if done % 200 == 0:
            print(f'  {done}/{N_APPS} scored...')

async_pool.shutdown(wait=False)
score_wall = time.time() - score_t0
sl = sorted(score_latencies)
print(f'\nScoring complete:')
print(f'  Scored: {len(results):,} | Errors: {score_errors:,}')
print(f'  Wall time: {score_wall:.1f}s | Throughput: {len(results)/max(score_wall,0.01):.0f} apps/s')
if sl:
    print(f'  Per-request latency:')
    print(f'    p50 : {sl[len(sl)//2]:>6.0f} ms')
    print(f'    p95 : {sl[int(len(sl)*0.95)]:>6.0f} ms')
    print(f'    p99 : {sl[int(len(sl)*0.99)]:>6.0f} ms')
    print(f'    mean: {sum(sl)/len(sl):>6.0f} ms')
if ingest_errors:
    print(f'\nIngest errors (first 3): {ingest_errors}')


In [ ]:
# Results analysis
df = pd.DataFrame(results)

if len(df) == 0:
    print('No results — all scoring requests failed. Check Gateway status.')
else:
    print('=== Decision Distribution ===')
    dec_counts = df['DECISION'].value_counts()
    for d in ['APPROVE', 'DECLINE']:
        n = dec_counts.get(d, 0)
        print(f'  {d:8s}: {n:>4d}  ({n/len(df)*100:.1f}%)')

    print('\n=== Breakdown by Application Type ===')
    cross = pd.crosstab(df['TYPE'], df['DECISION'], margins=True)
    print(cross.to_string())

    print('\n=== Detection Rate (suspicious flagged as DECLINE) ===')
    for app_type in ['suspicious_addr', 'suspicious_ip']:
        subset = df[df['TYPE'] == app_type]
        if len(subset) > 0:
            detected = subset[subset['DECISION'] == 'DECLINE']
            rate = len(detected) / len(subset) * 100
            print(f'  {app_type}: {len(detected)}/{len(subset)} ({rate:.1f}%)')

    clean = df[df['TYPE'] == 'clean']
    false_pos = clean[clean['DECISION'] == 'DECLINE']
    print(f'\n  False positive rate (clean -> DECLINE): {len(false_pos)}/{len(clean)} ({len(false_pos)/max(len(clean),1)*100:.1f}%)')


In [ ]:
# Post-simulation cost delta
# Allow ACCOUNT_USAGE views time to catch up (up to ~45 min lag)
post_credits = session.sql('''
    SELECT SUM(CREDITS_USED) AS TOTAL_CREDITS
    FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_HISTORY
    WHERE START_TIME >= DATEADD('hour', -2, CURRENT_TIMESTAMP())
''').collect()
credits_after = float(post_credits[0]['TOTAL_CREDITS'] or 0)
delta = credits_after - credits_before

print('=== Cost Estimate ===')
print(f'  Credits before: {credits_before:.4f}')
print(f'  Credits after : {credits_after:.4f}')
print(f'  Delta         : {delta:.4f} credits')
print(f'  Est. cost     : ${delta * 3.00:.4f} (at $3/credit)')
print()
print('Note: ACCOUNT_USAGE views have up to 45 min lag.')
print('For real-time cost, check ORGANIZATION_USAGE or RESOURCE_MONITORS.')

In [ ]:
# Latency summary table
def pct(arr, p):
    s = sorted(arr)
    return s[min(int(len(s) * p / 100), len(s) - 1)]

print('=== Latency Summary (ms) ===')
print(f'{"":<20} {"p50":>8} {"p95":>8} {"p99":>8} {"mean":>8}')
print(f'{"─" * 52}')
if score_latencies:
    print(f'{"Gateway (per app)":<20} {pct(score_latencies,50):>8.0f} {pct(score_latencies,95):>8.0f} {pct(score_latencies,99):>8.0f} {np.mean(score_latencies):>8.0f}')

print(f'\nScoring wall time: {score_wall:.1f}s ({len(results)} apps, {N_WORKERS} workers)')


## Summary

This notebook demonstrated the full end-to-end pipeline for 1000 synthetic tax applications:

1. **Synthetic data** — 70% clean, 20% sharing a fraud-ring address, 10% sharing a fraud-ring IP
2. **Stream ingest** — Pushed events via the OFS REST Ingest API (`/api/v1/ingest`) in batches of 50
3. **Real-time scoring** — Feature lookup (OFS online store) + Gateway inference with 10 concurrent workers
4. **Decisions** — APPROVE (<0.3), REVIEW (0.3–0.7), DECLINE (>0.7) based on fraud probability

**Key metrics:** detection rate for suspicious applications, false-positive rate for clean ones,
p50/p95/p99 latencies for ingest and inference, and estimated credit cost.

Next: Use `04_monitor.ipynb` to track model drift over time.